# Evaluation, A/B Testing & Monitoring

Offline metrics lie; online metrics are slow and expensive. This note covers temporal splits for offline eval, ML-specific A/B testing pitfalls, drift detection with PSI/KL, and dashboard design for production ML systems.

## What Interviewers Test
- Why random splits overestimate offline performance
- ML-specific A/B pitfalls: novelty effects, network effects, holdouts
- Drift detection: PSI and KS test implementations
- Feedback loops and the popularity bias spiral
- Dashboard design: what to alert on vs monitor

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
np.random.seed(42)

# --- Random split leakage demo ---
n = 2000
t = np.arange(n)
# Simulate temporal signal: user behavior drifts over time
true_signal = np.sin(t / 200) + 0.5 * np.random.randn(n)
y = (true_signal > 0).astype(int)
X = np.column_stack([true_signal + 0.1*np.random.randn(n), np.random.randn(n)])

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# Random split (WRONG for temporal data)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
lr = LogisticRegression().fit(X_tr, y_tr)
auc_random = roc_auc_score(y_te, lr.predict_proba(X_te)[:,1])

# Temporal split (CORRECT)
split = int(0.8 * n)
X_tr_t, X_te_t = X[:split], X[split:]
y_tr_t, y_te_t = y[:split], y[split:]
lr_t = LogisticRegression().fit(X_tr_t, y_tr_t)
auc_temporal = roc_auc_score(y_te_t, lr_t.predict_proba(X_te_t)[:,1])

print("=== Why Random Splits Lie on Temporal Data ===")
print(f"Random split AUC:   {auc_random:.4f}  (leaks future signal into training)")
print(f"Temporal split AUC: {auc_temporal:.4f}  (true out-of-time performance)")
print(f"Overestimation: {(auc_random - auc_temporal):.4f}")


## ML-Specific A/B Testing Pitfalls

| Pitfall | Description | Mitigation |
|---|---|---|
| **Novelty effect** | Users engage more just because it's new | Hold A/B for ≥ 2 weeks; check decay |
| **Network effect** | Users in same social graph affect each other | Cluster-based randomization |
| **Long-term holdout** | Short A/B misses long-term impact | Maintain 1–5% holdout for weeks |
| **Metric sensitivity** | Key metric moves but may not be statistically significant | Pre-register metrics; avoid p-hacking |
| **Survivorship bias** | Only active users return to test | Measure new user cohorts separately |

> 💡 **Interview Tip:** Interviewers love the novelty effect question. Answer: run the experiment for 2+ weeks, plot engagement over time, and look for a spike that decays back to baseline — that's novelty, not real improvement.


In [ ]:
# --- PSI (Population Stability Index) drift detection ---
def psi(expected, actual, n_bins=10):
    """
    PSI < 0.1: no significant shift
    PSI 0.1-0.2: moderate shift, investigate
    PSI > 0.2: significant shift, retrain
    """
    bins = np.linspace(min(expected.min(), actual.min()),
                       max(expected.max(), actual.max()) + 1e-6, n_bins+1)
    exp_pct = np.histogram(expected, bins)[0] / len(expected)
    act_pct = np.histogram(actual,   bins)[0] / len(actual)
    exp_pct = np.clip(exp_pct, 1e-6, None)
    act_pct = np.clip(act_pct, 1e-6, None)
    return np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))

# Simulate feature drift over time
train_dist  = np.random.normal(0, 1, 5000)
stable_dist = np.random.normal(0.1, 1, 2000)   # small shift
drifted_dist= np.random.normal(1.5, 1.5, 2000)  # large shift

psi_stable  = psi(train_dist, stable_dist)
psi_drifted = psi(train_dist, drifted_dist)

print(f"PSI (stable distribution):  {psi_stable:.4f} — {'OK' if psi_stable < 0.1 else 'INVESTIGATE'}")
print(f"PSI (drifted distribution): {psi_drifted:.4f} — {'RETRAIN' if psi_drifted > 0.2 else 'OK'}")

# KS test as complementary check
ks_stat_stable,  ks_p_stable  = stats.ks_2samp(train_dist, stable_dist)
ks_stat_drifted, ks_p_drifted = stats.ks_2samp(train_dist, drifted_dist)
print(f"KS stable:  stat={ks_stat_stable:.4f}, p={ks_p_stable:.4f}")
print(f"KS drifted: stat={ks_stat_drifted:.4f}, p={ks_p_drifted:.4f}")


## Feedback Loops & Popularity Bias

The popularity bias spiral:
1. System recommends popular items → users click on them
2. Model trains on click data → popular items get higher scores
3. System recommends even more popular items → long-tail items disappear

**Mitigations:** Exploration budget (ε-greedy or Thompson sampling), regularization toward item priors, diversity constraints, counterfactual evaluation (estimate what would have happened without bias).


## Monitoring Dashboard Design

| Layer | Metric | Alert threshold |
|---|---|---|
| **System health** | Latency p99, error rate, QPS | p99 > 200ms; error > 0.1% |
| **Data quality** | PSI per feature, null rate | PSI > 0.2; null > 2x baseline |
| **Model** | Score mean/std, prediction distribution | Shift > 2σ from 7d avg |
| **Business** | CTR, DAU, conversion | Drop > 5% vs 7d avg |
| **Labels** | Label rate, label delay | Rate < 50% of expected |

**Alert fatigue rule:** Alert on the 2–3 highest-signal metrics with clear thresholds. Everything else goes to dashboards only.


## Common Interview Questions

**Q: Why are random splits misleading for time-series ML problems?**
With a random split, training data contains events from the future relative to some test events. For models that capture temporal patterns (user behavior trends, seasonal effects), this leaks future signal into training. Always split chronologically: train on past, test on future.

**Q: What is PSI and what thresholds should you use?**
Population Stability Index measures how much a distribution has changed from reference to current. PSI = Σ (actual% - expected%) × log(actual%/expected%). PSI < 0.1: stable; 0.1–0.2: moderate shift, investigate; > 0.2: significant shift, consider retraining. These thresholds come from credit risk practice and are widely used in ML monitoring.

**Q: How do you detect and handle feedback loops?**
Monitor item diversity in recommendations over time — if the long tail disappears, you have a popularity bias spiral. Mitigations: add an exploration budget (recommend random or diverse items for some fraction of traffic), counterfactual logging, and regularize model scores toward popularity-free item quality signals.

**Q: What is a holdout group and when do you use it?**
A holdout group is a set of users permanently excluded from new model experiments, receiving only the control experience. They let you measure the cumulative long-term effect of changes without novelty effects contaminating measurement. Used when you expect significant long-term effects (behavior change, learning) that short A/B tests can't capture.

## Key Takeaways
- Random splits on temporal data overestimate performance; always use chronological splits
- A/B pitfalls: novelty effect (run 2+ weeks), network effects (cluster randomization), long-term holdouts
- PSI > 0.2 = significant drift, retrain; KS test as complementary check
- Feedback loops inflate popular items; fix with exploration budgets and diversity constraints
- Monitor 3 layers: system health, data quality, model/business metrics
- Alert on 2–3 high-signal metrics; route everything else to dashboards